# Experiment 2 — Identical means, different shapes: nonparametric separation

Figures per `docs/experiments.md`, Experiment 2 "Figures" subsection. This notebook only loads `results/exp2/results.parquet` and plots — it never recomputes algorithm results (see CLAUDE.md and `docs/experiments.md` §0.4). Run `python -m scripts.run_experiment2` (or `SMOKE_TEST=1 python -m scripts.run_experiment2` first) to generate it.

**REVISION NOTE (2026-09-14):** this notebook's figures were rewritten from per-family sweep line plots (τ/convergence/ARI vs. swept parameter, one plot per family) to a **6-setting bar-plot structure**, mirroring Experiment 3's "3 versions" blocks exactly. Rationale: the variation with the swept parameter within a family (as in Experiment 1) is not the interesting signal here — what matters is contrasting 6 distinct, illustrative instances (`scale_pair` σ_B∈{2,3}, `moment_matched_mixture` a∈{0.85,0.9}, `ring_blob` r∈{2,3}), each as its own block. `configs/exp2.yaml`'s canonical grid was itself trimmed to exactly these 6 settings (see that config's GRID TRIM NOTE) — nothing is computed and left unplotted.

8 algorithm rows per (setting, rep): `kabc`, `unif-kkm-1x`, `kabc-0.5x`, `unif-kkm-0.5x`, `kabc-sg`, `kabc-sg-0.5x`, `acb`, `acb-0.5x` — each `-0.5x` row is its OWN base algorithm re-run at 0.5× that base's own realized τ, testing how conservative that algorithm's own stopping rule is; `unif-kkm-1x`/`unif-kkm-0.5x` are tied specifically to `kabc`'s τ. **Multiplier note (2026-09-14):** this was originally 0.9x; changed to 0.5x once direct per-rep inspection showed KABC's doubling-trick schedule quantizes its own realized τ to a small set of round-boundary checkpoints (the final round alone typically costs roughly half of the total realized τ), so a 0.9x cut cannot actually test "10% less budget" — it almost always lands in the same "one checkpoint short" bucket as a much larger cut would.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kabc.plotting import set_style, style_for, savefig, ALGO_ORDER, ALGO_COLORS, ALGO_LABELS

set_style()

path = Path("../results/exp2/results.parquet")
if not path.exists():
    raise FileNotFoundError(
        f"{path} not found. Run `python -m scripts.run_experiment2` (or with SMOKE_TEST=1) "
        "to generate it -- this notebook only plots existing results, it never recomputes them."
    )
df = pd.read_parquet(path)
print(f"loaded {len(df)} rows from {path}")

BASE_ALGOS = [a for a in ["kabc", "kabc-sg", "acb"] if a in df["algorithm"].unique()]
assert len(BASE_ALGOS) == 3, f"expected 3 base algorithm rows, got {len(BASE_ALGOS)}: {BASE_ALGOS}"
ALL_ALGOS = [a for a in ALGO_ORDER if a in df["algorithm"].unique()]
assert len(ALL_ALGOS) == 8, f"expected 8 algorithm rows, got {len(ALL_ALGOS)}: {ALL_ALGOS}"

# 6 illustrative settings, NOT a parameter sweep -- mirrors Experiment 3's
# "3 versions" block structure exactly (see REVISION NOTE above). Each
# entry is (family, swept_value); SETTING_LABELS is the matching x-tick text.
SETTINGS = [
    ("scale_pair", 2), ("scale_pair", 3),
    ("moment_matched_mixture", 0.85), ("moment_matched_mixture", 0.9),
    ("ring_blob", 2), ("ring_blob", 3),
]
SETTING_LABELS = [
    "scale_pair\n$\\sigma_B$=2", "scale_pair\n$\\sigma_B$=3",
    "moment_matched\n$a$=0.85", "moment_matched\n$a$=0.90",
    "ring_blob\n$r$=2", "ring_blob\n$r$=3",
]


def setting_df(family, swept_value):
    return df[(df["family"] == family) & (df["swept_value"] == swept_value)]


df.head()


## Headline: what each of the 6 settings actually looks like

Purely illustrative reconstructions, one panel per setting, in a single row — not a recompute of algorithm results (same principle as the old design's "Figure 0"). `scale_pair`/`moment_matched_mixture` panels are exact analytic densities; `ring_blob` panels are real sample scatters from `RingBlobEnv` itself, so they stay in sync with the actual data-generating code.


In [ ]:
from scipy.stats import norm

from kabc.environments import RingBlobEnv

x = np.linspace(-6, 6, 500)
fig, axes = plt.subplots(1, 6, figsize=(19, 3.2))

for ax, (family, value), label in zip(axes, SETTINGS, SETTING_LABELS):
    if family == "scale_pair":
        ax.plot(x, norm.pdf(x, 0, 1), color="black", linewidth=1.6, label="A: N(0,1)")
        ax.plot(x, norm.pdf(x, 0, value), color="#1f77b4", linestyle="--", linewidth=1.6,
                label=f"B: N(0,{value}²)")
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.legend(fontsize=6.5, frameon=False)
    elif family == "moment_matched_mixture":
        b = np.sqrt(1 - value**2)
        density = 0.5 * norm.pdf(x, -value, b) + 0.5 * norm.pdf(x, value, b)
        ax.plot(x, norm.pdf(x, 0, 1), color="black", linewidth=1.6, label="A: N(0,1)")
        ax.plot(x, density, color="#d95f02", linestyle="--", linewidth=1.6, label=f"B: a={value}")
        ax.set_xlabel("x")
        ax.set_ylabel("density")
        ax.legend(fontsize=6.5, frameon=False)
    else:  # ring_blob -- pull(i) for i in [0,100) is the ring, [100,200) is the blob
        rng = np.random.default_rng(0)
        env = RingBlobEnv(N=200, ring_radius=value, ring_noise=0.1)
        pts = np.array([env.pull(i, rng) for i in range(200)])
        ax.scatter(pts[100:, 0], pts[100:, 1], s=6, color="black", alpha=0.5, label="blob")
        ax.scatter(pts[:100, 0], pts[:100, 1], s=6, color="#2ca02c", alpha=0.6, label=f"ring r={value}")
        ax.set_xlabel("x1")
        ax.set_ylabel("x2")
        ax.set_aspect("equal")
        ax.legend(fontsize=6.5, frameon=False)
    ax.set_title(label.replace("\n", " "), fontsize=9)

fig.tight_layout()
savefig(fig, "exp2_fig0_headline_settings", fig_dir="../results/figures")
plt.show()


## Figure 1: stopping budget (τ), all 6 settings × kabc/kabc-sg/acb/theoretical bound

One bar plot, 6 blocks (one per setting), 4 bars per block: `kabc`, `kabc-sg`, `acb` (mean τ ± std over reps) plus a `theoretical bound` bar (the grid point's formula-based `budget_cap` — a single deterministic value shared by all reps there, so no error bar). Mirrors Experiment 3's grouped-bar-plot convention exactly, with settings in place of versions.


In [ ]:
def budget_bar_plot(fig_name):
    n_bars = len(BASE_ALGOS) + 1  # +1 for the theoretical-bound bar
    bar_width = 0.8 / n_bars
    group_gap = 0.6

    fig, ax = plt.subplots(figsize=(12, 4))
    xtick_positions = []
    for s_idx, (family, value) in enumerate(SETTINGS):
        sub = setting_df(family, value)
        group_center = s_idx * (1.0 + group_gap)
        for a_idx, algo in enumerate(BASE_ALGOS):
            s = sub[sub["algorithm"] == algo]["tau"]
            x = group_center + (a_idx - (n_bars - 1) / 2) * bar_width
            ax.bar(x, s.mean(), yerr=s.std(), width=bar_width * 0.92, capsize=2,
                   color=ALGO_COLORS[algo], alpha=0.88,
                   label=ALGO_LABELS[algo] if s_idx == 0 else None)
        # Theoretical bound: restricted to BASE_ALGOS rows -- the "budget_cap"
        # column stores each row's OWN used budget, and derived -0.5x/
        # unif-kkm rows store a reduced value, not the grid point's true
        # formula-based bound (same fix as notebook 01's theory overlay).
        # It's identical across all base rows/reps at a setting, so no error bar.
        bound = sub[sub["algorithm"].isin(BASE_ALGOS)]["budget_cap"].mean()
        x = group_center + (len(BASE_ALGOS) - (n_bars - 1) / 2) * bar_width
        ax.bar(x, bound, width=bar_width * 0.92, color="grey", alpha=0.6,
               label="theoretical bound" if s_idx == 0 else None)
        xtick_positions.append(group_center)

    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(SETTING_LABELS, fontsize=8)
    ax.set_ylabel(r"mean $\tau$ (± std)")
    ax.set_yscale("log")
    ax.legend(fontsize=7, frameon=False, ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.15))
    fig.tight_layout()
    savefig(fig, fig_name, fig_dir="../results/figures")
    plt.show()


budget_bar_plot("exp2_fig1_budget_bars")


## Figure 2: convergence rate, all 6 settings × kabc/kabc-sg/acb

Fraction of reps with `stopped=True`, mean ± std across reps (a Bernoulli mean ± its std). Read alongside Figure 1 and Figure 3: a method that never converges reports its graceful trivial-cluster fallback (ARI exactly 0.0), not a real clustering attempt.


In [ ]:
def grouped_bar_plot(algos, metric_col, ylabel, fig_name, log_scale=False, legend_ncol=None):
    """One bar plot: 6 sets (settings) x len(algos) bars, grouped with a
    visible gap between sets -- identical convention to Experiment 3's
    grouped_bar_plot, with settings in place of versions."""
    n_algo = len(algos)
    bar_width = 0.8 / n_algo
    group_gap = 0.6

    fig, ax = plt.subplots(figsize=(12, 4))
    xtick_positions = []
    for s_idx, (family, value) in enumerate(SETTINGS):
        sub = setting_df(family, value)
        group_center = s_idx * (1.0 + group_gap)
        for a_idx, algo in enumerate(algos):
            s = sub[sub["algorithm"] == algo][metric_col]
            x = group_center + (a_idx - (n_algo - 1) / 2) * bar_width
            ax.bar(x, s.mean(), yerr=s.std(), width=bar_width * 0.92, capsize=2,
                   color=ALGO_COLORS[algo], alpha=0.88,
                   label=ALGO_LABELS[algo] if s_idx == 0 else None)
        xtick_positions.append(group_center)

    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(SETTING_LABELS, fontsize=8)
    ax.set_ylabel(ylabel)
    if log_scale:
        ax.set_yscale("log")
    ax.legend(fontsize=7, frameon=False, ncol=legend_ncol or n_algo,
              loc="upper center", bbox_to_anchor=(0.5, 1.15))
    fig.tight_layout()
    savefig(fig, fig_name, fig_dir="../results/figures")
    plt.show()


grouped_bar_plot(BASE_ALGOS, "stopped", "convergence rate (± std)", "exp2_fig2_convergence_bars")


## Figure 3: ARI, all 6 settings × all 8 algorithm rows

Mean ± std over reps, all 8 rows per block. As in Experiment 1: a SMALL ARI drop from a base algorithm to its own `-0.5x` variant means little margin was needed; a LARGE drop means real accuracy was still being bought in that last half of budget. `unif-kkm-1x`/`unif-kkm-0.5x` sitting close to `kabc`/`kabc-0.5x` would suggest active/adaptive selection isn't earning much of an edge at that matched budget.


In [ ]:
grouped_bar_plot(ALL_ALGOS, "ari", "mean ARI (± std)", "exp2_fig3_ari_bars", legend_ncol=4)


## Summary tables: exact mean ± std per algorithm, per setting

Printed (not just plotted), one table per setting (6 total), same convention as notebook 01/03's summary tables.


In [ ]:
for family, value in SETTINGS:
    sub = setting_df(family, value)
    summary = (
        sub.groupby("algorithm")
        .agg(
            mean_tau=("tau", "mean"), std_tau=("tau", "std"),
            mean_ari=("ari", "mean"), std_ari=("ari", "std"),
            frac_stopped=("stopped", "mean"), std_stopped=("stopped", "std"),
        )
        .reset_index()
    )
    summary["algorithm"] = pd.Categorical(summary["algorithm"], categories=ALL_ALGOS, ordered=True)
    summary = summary.sort_values("algorithm")
    print(f"=== {family} @ {value} ===")
    display(summary)
